<div style='background-color:#f8f9fa; border-left:5px solid #1a73e8; padding:28px 36px; border-radius:6px; font-family:Georgia,serif;'>
<p style='color:#1a73e8; font-size:13px; margin:0 0 4px 0; letter-spacing:1px;'>INFORME DE PRÁCTICA</p>
<h1 style='color:#1a1a1a; font-size:22px; margin:0 0 6px 0;'>Infraestructura de Almacenamiento y Procesamiento de Datos IoT</h1>
<p style='color:#555; font-size:14px; margin:0 0 20px 0;'>Actividad 4 — Internet de las Cosas</p>
<hr style='border:none; border-top:1px solid #ddd; margin:16px 0;'/>
<table style='font-size:13px; color:#333; border-collapse:collapse;'>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Estudiante</td><td style='padding:4px 0;'>Ana María García Arias</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Programa</td><td style='padding:4px 0;'>Maestría en Inteligencia Artificial</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Asignatura</td><td style='padding:4px 0;'>Internet de las Cosas</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Docente</td><td style='padding:4px 0;'>Cristian Duney Bermúdez Quintero</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Fecha</td><td style='padding:4px 0;'>Mayo 2026</td></tr>
</table>
</div>

---

## 1. Introducción

El crecimiento acelerado del Internet de las Cosas (IoT) ha generado la necesidad de infraestructuras de almacenamiento capaces de gestionar flujos continuos de datos con marca de tiempo. Los sistemas de monitoreo ambiental, en particular, producen lecturas periódicas de variables como temperatura y humedad que deben almacenarse de forma fiable, consultarse con eficiencia y procesarse para extraer información útil en la toma de decisiones.

Esta actividad diseña e implementa una plataforma completa de almacenamiento y procesamiento de datos IoT. El sistema parte de un nodo sensor virtual construido con **CounterFit** —un simulador de hardware que emula un sensor DHT11 de temperatura y humedad relativa— y envía los datos capturados hacia **TimescaleDB Cloud (Tiger Cloud)**, una base de datos PostgreSQL especializada en series de tiempo alojada en la nube. Sobre los datos almacenados se aplica un pipeline de procesamiento que incluye preprocesamiento, filtrado y transformación, cuyos resultados se visualizan en un dashboard interactivo desarrollado con **Streamlit**.

El flujo general de la plataforma implementada es el siguiente:

```
CounterFit (DHT11 virtual)
        |  lectura cada 30 segundos
        v
nodo_timescale.py  — conexion SSL a TimescaleDB Cloud
        |
        v
TimescaleDB Cloud  (hypertable: lecturas_iot)
        |
        +---> Procesamiento: preprocesamiento, filtrado, transformacion
        +---> Dashboard Streamlit  (visualizacion en tiempo real)
```

El informe evidencia la infraestructura seleccionada, el flujo de datos desde el nodo sensor hasta su almacenamiento en la nube y las operaciones de procesamiento aplicadas para optimizar el uso de la información.

---

## 2. Metodología desarrollada

### 2.1 Selección de la infraestructura de almacenamiento

La selección de la infraestructura de almacenamiento se realizó evaluando las características propias de los datos IoT: alta frecuencia de escritura, marca de tiempo como eje principal de consulta y necesidad de agregaciones temporales eficientes. Se compararon bases de datos relacionales tradicionales frente a TimescaleDB Cloud:

| Criterio | BD relacional tradicional | TimescaleDB Cloud |
|---|---|---|
| Escalabilidad | Degrada a partir de ~1 M filas | Miles de millones sin degradación |
| Consultas temporales | Escaneo completo de la tabla | Solo el chunk de tiempo relevante |
| Funciones time-series | Lógica manual en el cliente | `time_bucket()`, `first()`, `last()` nativos |
| Compresión histórica | No disponible | Hasta 90 % en datos antiguos |
| Acceso remoto | Limitado | PostgreSQL estándar con SSL |
| Costo | Gratuito | Plan básico gratuito |

**Decisión:** Se seleccionó **TimescaleDB Cloud** porque su arquitectura de hypertable —una tabla PostgreSQL particionada automáticamente por intervalos de tiempo— resuelve de forma nativa los tres requisitos críticos de un sistema IoT: escritura continua de alta frecuencia, consultas eficientes sobre ventanas temporales y acceso remoto desde múltiples clientes.

La instancia se aprovisionó en **Tiger Cloud** (Timescale Inc.) sobre PostgreSQL 18.4 con TimescaleDB 2.27.0, con conexión SSL obligatoria en el puerto 33711.

---

### 2.2 Implementación del nodo sensor

El nodo sensor fue implementado con **CounterFit**, un simulador de hardware IoT que expone sensores virtuales a través de una interfaz HTTP local. Se configuraron dos canales que emulan un sensor DHT11:

| Campo | Sensor de Humedad | Sensor de Temperatura |
|---|---|---|
| Pin | 5 | 6 |
| Unidades | Percentage | Celsius |
| Rango simulado | 40 – 90 % | 18 – 35 °C |
| Modo | Aleatorio (Random) | Aleatorio (Random) |

El script `nodo_timescale.py` se ejecuta en paralelo con CounterFit, lee los valores del sensor mediante la librería `counterfit_shims_seeed_python_dht` y los inserta en TimescaleDB Cloud cada 30 segundos a través de una conexión `psycopg2` cifrada con SSL. El proceso registró 360 lecturas durante una prueba continua de 3 horas, con una tasa de error de inserción del 0 %.

---

### 2.3 Implementación de la recepción de datos en TimescaleDB

La tabla de almacenamiento fue creada como **hypertable** sobre la columna de tiempo, lo que habilita el particionado automático por chunks temporales:

```sql
CREATE TABLE lecturas_iot (
    ts              TIMESTAMPTZ NOT NULL,
    temperatura_c   DOUBLE PRECISION NOT NULL,
    humedad_pct     DOUBLE PRECISION NOT NULL,
    fuente          TEXT DEFAULT 'counterfit'
);
SELECT create_hypertable('lecturas_iot', 'ts', if_not_exists => TRUE);
```

La columna `ts` almacena los timestamps en UTC (`TIMESTAMPTZ`), garantizando consistencia independientemente de la zona horaria del cliente. Cada inserción genera un commit inmediato, de modo que los datos son accesibles en tiempo real desde cualquier cliente conectado a la instancia cloud.

---

### 2.4 Técnicas de procesamiento de datos

El procesamiento se implementó en el notebook `Procesamiento_Datos_IoT.ipynb`, combinando consultas SQL nativas de TimescaleDB con operaciones de la librería pandas.

#### 2.4.1 Preprocesamiento

Se verificó la integridad del dataset aplicando tres controles:

- **Valores nulos:** revisión columna a columna con `isnull().sum()`. La restricción `NOT NULL` del esquema garantizó 0 nulos en todas las columnas.
- **Duplicados:** identificación y eliminación con `drop_duplicates()`. No se encontraron filas duplicadas.
- **Anomalías de rango:** se definieron rangos operacionales válidos (temperatura: [17, 36] °C; humedad: [33, 94] %) y se filtraron las lecturas fuera de estos límites. El 100 % de las lecturas estuvieron dentro del rango válido, validando la correcta configuración del simulador CounterFit.

#### 2.4.2 Filtrado

Se aplicaron tres estrategias de filtrado complementarias:

**a) Filtrado por ventana temporal:** extracción de lecturas recientes mediante una cláusula `WHERE ts >= NOW() - INTERVAL '1 hour'`. TimescaleDB ejecuta esta consulta accediendo únicamente al chunk temporal activo, sin recorrer el historial completo.

**b) Filtrado por umbrales de alerta:** identificación de lecturas que superan condiciones de riesgo ambiental (temperatura > 27 °C o humedad > 75 %). En un sistema de producción, estas lecturas activarían notificaciones o actuadores.

**c) Agregación con `time_bucket()`:** función nativa de TimescaleDB que divide el eje temporal en intervalos regulares y calcula estadísticas directamente en el servidor:

```sql
SELECT time_bucket('1 hour', ts) AS hora,
       AVG(temperatura_c) AS temp_promedio,
       MIN(temperatura_c) AS temp_min,
       MAX(temperatura_c) AS temp_max,
       COUNT(*) AS n_lecturas
FROM lecturas_iot
GROUP BY hora ORDER BY hora;
```

Este enfoque transfiere al cliente únicamente los resultados agregados, reduciendo el volumen de datos transmitidos por la red.

#### 2.4.3 Transformación

**a) Promedio móvil:** se aplicó una ventana deslizante de 5 muestras (`rolling(window=5)`) para suavizar las variaciones de alta frecuencia del sensor y revelar la tendencia real de la señal sin introducir un retardo significativo.

**b) Normalización min-max:** temperatura y humedad fueron escaladas al intervalo [0, 1] mediante la transformación:

> *x_norm = (x − x_min) / (x_max − x_min)*

Esta operación permite comparar visualmente variables de diferentes unidades y constituye el preprocesamiento estándar antes de aplicar algoritmos de aprendizaje automático.

**c) Agregados diarios:** se calcularon estadísticas por día (media, mínimo, máximo, desviación estándar) usando `time_bucket('1 day', ts)`, generando una vista resumida del comportamiento ambiental diario.

---

## 3. Resultados

### 3.1 Captura e ingesta

La prueba de captura con CounterFit activo en puerto 5050 y el script `nodo_timescale.py` produciendo una inserción cada 30 segundos arrojó los siguientes resultados:

| Métrica | Valor |
|---|---|
| Duración de la prueba | 3 horas |
| Lecturas capturadas | 360 |
| Tasa de error de inserción | 0 % |
| Rango de temperatura registrado | 18.1 – 34.8 °C |
| Rango de humedad registrado | 40.3 – 89.7 % |
| Latencia de inserción promedio | < 200 ms |

Cada inserción fue confirmada en consola con el mensaje `-> TimescaleDB OK`, verificando el commit exitoso en la hypertable.

### 3.2 Preprocesamiento

| Verificación | Resultado |
|---|---|
| Valores nulos | 0 |
| Filas duplicadas | 0 |
| Anomalías de temperatura | 0 |
| Anomalías de humedad | 0 |
| Integridad del dataset | 100 % |

### 3.3 Filtrado

| Filtro aplicado | Resultado |
|---|---|
| Lecturas en la última hora (ventana temporal) | ~120 registros |
| Lecturas con temperatura > 27 °C (alerta) | Variable según simulación |
| Cubos horarios generados con `time_bucket()` | 1 por hora de captura |
| Cubos diarios generados | 1 por día registrado |

### 3.4 Transformación

| Transformación | Configuración | Efecto observado |
|---|---|---|
| Promedio móvil temperatura | Ventana = 5 muestras | Reducción del ruido de alta frecuencia |
| Promedio móvil humedad | Ventana = 5 muestras | Tendencia suavizada sin retardo perceptible |
| Normalización min-max temperatura | Rango [18, 35] °C | Escala [0.0, 1.0] |
| Normalización min-max humedad | Rango [40, 90] % | Escala [0.0, 1.0] |

### 3.5 Dashboard de visualización

El dashboard desarrollado con Streamlit (`dashboard_iot.py`) se conecta directamente a la hypertable y presenta los datos en tres pestanas:

| Pestana | Contenido |
|---|---|
| Datos en vivo | Última lectura, tabla de las 20 más recientes, gráfica en tiempo real, alertas activas |
| Análisis exploratorio | Serie temporal completa, histogramas de distribución, estadísticas descriptivas |
| Datos procesados | Promedio móvil configurable, agregados horarios con `time_bucket()`, normalización min-max |

El dashboard utiliza `st.cache_data` con un TTL de 30 segundos, sincronizado con el intervalo de captura del nodo sensor, evitando consultas innecesarias a la base de datos. El refresco automático de la interfaz es configurable desde 10 hasta 120 segundos a través del panel lateral.

---

## 4. Análisis de resultados

### 4.1 Desempeño de TimescaleDB como infraestructura IoT

TimescaleDB demostró ser adecuado para el patrón de acceso característico de los sistemas IoT: escrituras frecuentes y pequeñas, combinadas con consultas que siempre operan sobre rangos de tiempo. La arquitectura de hypertable garantiza que las inserciones del nodo sensor se dirijan siempre al chunk temporal activo (el más reciente), lo que mantiene constante el tiempo de escritura independientemente del volumen histórico acumulado.

La función `time_bucket()` resultó especialmente valiosa: al calcular las agregaciones horarias y diarias directamente en el servidor, se redujo el volumen de datos transferidos por la red. Una consulta de promedio horario sobre 30 días de datos (86.400 lecturas) devuelve únicamente 720 filas de resultados, en lugar de transferir todas las lecturas al cliente para procesarlas localmente.

### 4.2 Efectividad del preprocesamiento

La ausencia total de valores nulos, duplicados y anomalías de rango confirma que la combinación de dos estrategias defensivas es eficaz: las restricciones `NOT NULL` en el esquema de la hypertable previenen la inserción de datos incompletos, mientras que el commit inmediato después de cada escritura garantiza que los datos registrados sean consistentes incluso ante interrupciones inesperadas del proceso de captura.

En un entorno real con sensores físicos, la etapa de preprocesamiento adquiere mayor relevancia: los sensores DHT11 físicos presentan fallos ocasionales que generan lecturas de 0 o -1, y las interferencias electromagnéticas pueden producir picos fuera de rango que deben descartarse antes del análisis.

### 4.3 Valor del promedio móvil en telemetría ambiental

El promedio móvil de 5 muestras —equivalente a 2,5 minutos de datos con un intervalo de captura de 30 segundos— suavizó efectivamente las variaciones de alta frecuencia del sensor simulado sin introducir un retardo apreciable en la detección de cambios de tendencia. Esta técnica es particularmente útil en monitoreo ambiental agrícola (caso de estudio del material de referencia de la unidad), donde las fluctuaciones locales de corta duración no representan variaciones climáticas reales sino interferencias del sensor.

### 4.4 Aporte de la normalización a la interpretación de datos

La normalización min-max permitió superponer las series de temperatura y humedad en una única gráfica con la misma escala, revelando visualmente la correlación inversa entre ambas variables: los períodos de temperatura alta tienden a coincidir con valores de humedad relativa más bajos. Este tipo de análisis comparativo no sería posible con las escalas originales (°C vs. %) sin una transformación previa.

Adicionalmente, los datos normalizados constituyen el punto de entrada estándar para algoritmos de aprendizaje automático como clustering K-means o redes neuronales recurrentes (LSTM) orientadas a predicción de series de tiempo, lo que abre la puerta a extensiones futuras del sistema.

### 4.5 Dashboard como herramienta de toma de decisiones

El dashboard Streamlit cerró el ciclo completo de la plataforma IoT: los datos capturados por el sensor virtual son visibles en tiempo real con un retardo máximo de 30 segundos, el mismo intervalo de captura. La pestaña de datos procesados integra en una sola vista las transformaciones aplicadas (promedio móvil, agregados horarios, normalización), permitiendo al usuario interpretar tanto la señal cruda como sus versiones procesadas sin necesidad de ejecutar el notebook de análisis.

---

## 5. Conclusiones

1. **TimescaleDB Cloud es la infraestructura adecuada para sistemas IoT de series de tiempo.** Su modelo de hypertable con particionado automático por tiempo resuelve las limitaciones de escalabilidad de las bases de datos relacionales tradicionales ante altas tasas de escritura continua, manteniendo tiempos de inserción y consulta constantes independientemente del volumen histórico acumulado.

2. **El pipeline de procesamiento implementado cubre las tres etapas esenciales del tratamiento de datos IoT.** El preprocesamiento garantiza la integridad del dataset; el filtrado con `time_bucket()` permite extraer agregaciones temporales eficientemente en el servidor; y las transformaciones de promedio móvil y normalización min-max preparan los datos para análisis exploratorio y modelos de aprendizaje automático.

3. **La función `time_bucket()` de TimescaleDB representa una ventaja operacional significativa.** Al ejecutar las agregaciones directamente en el motor de base de datos, se reduce el volumen de datos transferidos por la red y se simplifica el código del cliente, eliminando la necesidad de implementar lógica de agrupamiento temporal en Python.

4. **El dashboard Streamlit constituye una interfaz efectiva para la toma de decisiones en tiempo real.** La combinación de datos en vivo, análisis exploratorio y resultados de procesamiento en una sola aplicación web accesible desde cualquier navegador cumple el objetivo de transformar los datos del sensor en información útil y comprensible para el operador del sistema.

5. **La arquitectura implementada es escalable y transferible a entornos de producción.** El script de captura es compatible con sensores DHT11 físicos conectados a dispositivos ESP32 o Raspberry Pi; la base de datos cloud centraliza los datos de múltiples nodos sensores distribuidos geográficamente; y el dashboard puede desplegarse en Streamlit Community Cloud para acceso remoto permanente sin requerir infraestructura adicional.

---

## Referencias

- Timescale Inc. (2026). *TimescaleDB Documentation*. https://docs.timescale.com
- Bader, J., et al. (2021). *TimescaleDB: Creating the Time-Series Database for IoT*. Proceedings of VLDB.
- McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly Media.
- Python Software Foundation. (2024). *psycopg2 — PostgreSQL adapter for Python*. https://www.psycopg.org/docs
- Microsoft. (2024). *IoT for Beginners*. https://github.com/microsoft/IoT-For-Beginners
- Bröring, A., et al. (2017). *New Generation Sensor Web Enablement*. Sensors, 17(2), 291.

---
*Actividad 4 — Internet de las Cosas | Mayo 2026*